# Lab Exercise: Cloud Deployment Cost Analysis and Scaling Decisions
## AIAT 125 — Unit 3: Cloud Deployment and Infrastructure

**Learning objectives**
1. Calculate monthly cloud compute costs for different instance types.
2. Select the right instance type based on throughput and latency SLA requirements.
3. Compare vertical vs horizontal scaling strategies and their cost implications.
4. Build a complete deployment recommendation function.

**Why this matters**  
Cloud infrastructure costs money the moment it starts. Making the wrong instance choice can cost 5–10× more than necessary, or fail to meet your SLA. Choosing between scaling up (bigger machine) and scaling out (more machines) is a decision every ML engineer makes in production.

**Grading** — 100 points total
| Task | Points |
|---|---|
| Task 1: Monthly cost calculator | 25 |
| Task 2: Instance selector by SLA | 25 |
| Task 3: Horizontal scaling simulation | 25 |
| Task 4: Deployment recommendation report | 25 |

> This lab runs entirely locally — no cloud credentials needed.


In [ ]:
import math
import numpy as np

# Cloud instance catalog (AWS pricing, us-east-1, on-demand, $/hour)
INSTANCE_CATALOG = {
    "t3.medium"   : 0.0416,
    "t3.large"    : 0.0832,
    "m5.xlarge"   : 0.192,
    "g4dn.xlarge" : 0.526,
    "g4dn.2xlarge": 0.752,
}

# Benchmark data for a 50MB sklearn model: {instance: {max_rps, p95_latency_ms}}
INSTANCE_BENCHMARKS = {
    "t3.medium"   : {"max_rps": 20,   "p95_latency_ms": 450},
    "t3.large"    : {"max_rps": 50,   "p95_latency_ms": 190},
    "m5.xlarge"   : {"max_rps": 120,  "p95_latency_ms": 80},
    "g4dn.xlarge" : {"max_rps": 500,  "p95_latency_ms": 35},
    "g4dn.2xlarge": {"max_rps": 1000, "p95_latency_ms": 20},
}

print("Instance catalog:", list(INSTANCE_CATALOG.keys()))
print("Setup complete.")

---
## Task 1 — Monthly Cost Calculator (25 points)

Implement `calculate_monthly_cost(instance_type, hours_per_day, days_per_month)` → `float`.

Formula: `cost = price_per_hour × hours_per_day × days_per_month`

Validation rules (raise `ValueError` if violated):
- `instance_type` must be in `INSTANCE_CATALOG`
- `hours_per_day` must be between 0 and 24 (inclusive)
- `days_per_month` must be between 1 and 31 (inclusive)

In [ ]:
def calculate_monthly_cost(instance_type, hours_per_day=24, days_per_month=30):
    """
    Returns total monthly cost in USD for running an instance.
    Raises ValueError for invalid inputs.
    """
    # TODO: Validate instance_type (raise ValueError with message if not in catalog)
    # TODO: Validate hours_per_day (0 to 24)
    # TODO: Validate days_per_month (1 to 31)
    # TODO: Compute and return cost
    # YOUR CODE HERE
    pass

# --- Test: 24/7 production costs ---
print("Monthly cost (24h/day, 30 days):")
for instance in INSTANCE_CATALOG:
    cost = calculate_monthly_cost(instance, 24, 30)
    print(f"  {instance:15s}: ${cost:>8.2f}/month")

print("\nDev environment (8h/day, 22 working days):")
for instance in ["t3.medium", "t3.large"]:
    cost = calculate_monthly_cost(instance, 8, 22)
    print(f"  {instance:15s}: ${cost:>8.2f}/month")

# Validation
assert abs(calculate_monthly_cost("t3.medium", 24, 30) - (0.0416 * 24 * 30)) < 0.01
assert abs(calculate_monthly_cost("g4dn.xlarge", 8, 22) - (0.526 * 8 * 22)) < 0.01
try:
    calculate_monthly_cost("p4d.24xlarge")
    assert False, "Should raise ValueError for unknown instance"
except ValueError:
    pass
try:
    calculate_monthly_cost("t3.medium", hours_per_day=25)
    assert False, "Should raise ValueError for hours > 24"
except ValueError:
    pass
print("\nTask 1 PASSED")

---
## Task 2 — Instance Selector by SLA (25 points)

Implement `choose_instance(required_rps, latency_budget_ms)` that picks the **cheapest** instance that satisfies both:
- `INSTANCE_BENCHMARKS[instance]["max_rps"] >= required_rps`
- `INSTANCE_BENCHMARKS[instance]["p95_latency_ms"] <= latency_budget_ms`

Return `{"instance": name, "cost_per_hour": ..., "max_rps": ..., "p95_latency_ms": ...}` or `None` if nothing qualifies.

In [ ]:
def choose_instance(required_rps, latency_budget_ms):
    """
    Pick the cheapest instance meeting both RPS and latency SLA requirements.
    Returns a dict or None.
    """
    # TODO: Filter INSTANCE_BENCHMARKS to candidates meeting both conditions
    # TODO: From valid candidates, pick the one with the lowest cost_per_hour
    # TODO: Return the result dict (or None if no candidate qualifies)
    # YOUR CODE HERE
    pass

# --- Test across different SLA requirements ---
scenarios = [
    (10,  500, "Internal dashboard"),
    (40,  200, "Internal analytics API"),
    (80,  100, "Customer-facing API"),
    (400, 50,  "Real-time search feature"),
    (2000, 10, "High-frequency trading"),
]

print(f"{'Scenario':<28} {'RPS':>5} {'p95ms':>6} {'Chosen':>15} {'$/hr':>7}")
print("-" * 65)
for rps, lat, name in scenarios:
    result = choose_instance(rps, lat)
    if result:
        print(f"  {name:<26} {rps:>5} {lat:>6} "
              f"{result['instance']:>15} ${result['cost_per_hour']:>6.3f}")
    else:
        print(f"  {name:<26} {rps:>5} {lat:>6} {'No instance meets SLA':>22}")

# Validation
r1 = choose_instance(10, 500)
assert r1 is not None and r1["instance"] == "t3.medium", f"10rps/500ms → t3.medium, got {r1}"
r2 = choose_instance(80, 100)
assert r2 is not None and r2["instance"] == "m5.xlarge", f"80rps/100ms → m5.xlarge, got {r2}"
r3 = choose_instance(2000, 10)
assert r3 is None, "2000rps/10ms → None (no instance qualifies)"
print("\nTask 2 PASSED")

---
## Task 3 — Horizontal Scaling Simulation (25 points)

When traffic grows beyond a single instance's capacity, you add more replicas behind a load balancer (**horizontal scaling**).

Implement `simulate_scaling(current_instance, target_rps)` that:
1. Gets `max_rps_per_instance` from `INSTANCE_BENCHMARKS[current_instance]`
2. Calculates `replicas = ceil(target_rps / max_rps_per_instance)` using `math.ceil`
3. Calculates `total_cost_per_hour = replicas × INSTANCE_CATALOG[current_instance]`
4. Returns `{"strategy": "horizontal", "current_instance": ..., "replicas": ..., "total_rps_capacity": ..., "cost_per_hour": ...}`

Then compare horizontal-scale cost against the cheapest single larger instance.

In [ ]:
def simulate_scaling(current_instance, target_rps):
    """
    Simulate horizontal scaling of current_instance to reach target_rps.
    Returns a dict with scaling details.
    """
    # TODO: Get max_rps for current_instance
    # TODO: replicas = math.ceil(target_rps / max_rps_per_instance)
    # TODO: total_cost_per_hour = replicas * INSTANCE_CATALOG[current_instance]
    # TODO: Return dict
    # YOUR CODE HERE
    pass

# --- Compare horizontal scaling vs vertical (single larger instance) ---
scaling_tests = [
    ("t3.large", 200,  "t3.large → 200 rps"),
    ("t3.large", 500,  "t3.large → 500 rps"),
    ("m5.xlarge", 500, "m5.xlarge → 500 rps"),
]

print(f"{'Scenario':<30} {'Replicas':>8} {'H-scale $/hr':>14} {'Best single $/hr':>18}")
print("-" * 74)
for instance, target, label in scaling_tests:
    h = simulate_scaling(instance, target)
    single = choose_instance(target, 9999)  # any latency OK
    h_cost = h["cost_per_hour"] if h else float("inf")
    s_name = single["instance"] if single else "none"
    s_cost = single["cost_per_hour"] if single else float("inf")
    cheaper = "h-scale" if h_cost < s_cost else "single"
    print(f"  {label:<28} {h['replicas'] if h else '?':>8}"
          f"     ${h_cost:>7.3f}         ${s_cost:>7.3f} ({s_name})  ← {cheaper} cheaper")

# Validation
r = simulate_scaling("t3.large", 200)
assert r is not None and r["strategy"] == "horizontal"
assert r["replicas"] == math.ceil(200 / INSTANCE_BENCHMARKS["t3.large"]["max_rps"])
assert r["total_rps_capacity"] >= 200
print("\nTask 3 PASSED")

---
## Task 4 — Deployment Recommendation Report (25 points)

Implement `deployment_recommendation(model_size_mb, expected_rps, latency_budget_ms, budget_per_month_usd)` that returns a structured dict.

Required keys:
- `recommended_instance` (str or `None`)
- `monthly_cost_usd` (float, for 24h × 30 days)
- `within_budget` (bool)
- `expected_rps_capacity` (int)
- `p95_latency_ms` (float)
- `notes` (str) — human-readable explanation including: why this instance was chosen, budget fit, and a warning if `model_size_mb > 500` (large models may need GPU)

In [ ]:
def deployment_recommendation(model_size_mb, expected_rps, latency_budget_ms, budget_per_month_usd):
    """
    Generate a deployment recommendation.
    Returns a dict with the 6 required keys.
    """
    # TODO: Use choose_instance(expected_rps, latency_budget_ms) to find the right instance
    # TODO: Compute monthly_cost = calculate_monthly_cost(instance, 24, 30)
    # TODO: within_budget = monthly_cost <= budget_per_month_usd
    # TODO: Build a notes string that explains:
    #   - Which instance was selected and why (or "no instance meets SLA")
    #   - Whether it fits the budget
    #   - If model_size_mb > 500: warn that large models may require GPU instances
    # YOUR CODE HERE
    pass

# --- Run for 4 different scenarios ---
scenarios = [
    (50,   10,  500, 50,   "Dev chatbot prototype"),
    (200,  80,  100, 500,  "Production recommendation API"),
    (500,  400, 50,  2000, "Real-time image classification"),
    (2000, 5,   200, 30,   "Large LLM on tight budget"),
]

for model_mb, rps, latency, budget, name in scenarios:
    rec = deployment_recommendation(model_mb, rps, latency, budget)
    print(f"\n=== {name} ===")
    if rec:
        for k, v in rec.items():
            print(f"  {k}: {v}")
    else:
        print("  ERROR: function returned None")

# Validation
rec = deployment_recommendation(50, 10, 500, 100)
assert isinstance(rec, dict)
for k in ["recommended_instance", "monthly_cost_usd", "within_budget",
          "expected_rps_capacity", "p95_latency_ms", "notes"]:
    assert k in rec, f"Missing key: {k}"
assert rec["recommended_instance"] == "t3.medium"
assert rec["within_budget"] == True
print("\nTask 4 PASSED")

In [ ]:
# --- Final summary ---
print("=" * 55)
print("UNIT 3 LAB — FINAL GATE")
print("=" * 55)
checks = {
    "Task 1: cost calculator validates inputs": True,  # passed inline above
    "Task 2: instance selector picks cheapest valid": r1 is not None and r1["instance"] == "t3.medium",
    "Task 3: horizontal scaling returns correct replicas": r["replicas"] == math.ceil(200 / INSTANCE_BENCHMARKS["t3.large"]["max_rps"]),
    "Task 4: recommendation has all 6 keys": isinstance(rec, dict) and len(rec) >= 6,
}
for task, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {task}")
print("=" * 55)

---
## Self-Check Questions

1. **CapEx vs OpEx**: Cloud is pay-per-use (OpEx). When does buying your own hardware (CapEx) become cheaper?
2. **A model that runs on your laptop fails on a `t3.medium` in production.** Name 3 environment differences that could cause this.
3. **Horizontal vs vertical**: Your SLA requires p95 < 100ms. Traffic doubles. Should you scale to 2 × `m5.xlarge` or upgrade to `g4dn.xlarge`? Use Task 2 and 3 output to justify your answer.
4. **Why is `model_size_mb > 500` a signal to consider GPU instances?** What operation becomes the bottleneck on CPU for large neural networks?